In [ ]:
!pip install datasets
!pip install transformers evaluate datasets
!pip install torch
!pip install wandb
!pip install fuzzywuzzy python-Levenshtein

In [92]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, DefaultDataCollator, pipeline
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Tuple
from collections import Counter
from fuzzywuzzy import fuzz
from tqdm import tqdm
import torch.nn.functional as F
import numpy as np
import torch
import re
import string
import math

In [93]:
class SimpleContextRetriever:
  def __init__(self, retriever, batch_size=32):
    self.retriever = retriever
    self.batch_size = batch_size
    self.contexts = None
    self.context_embeddings = None

  def normalize_text(self, text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

  def calculate_bm25_scores(self, query, docs):
    tokenized_query = query.split()
    tokenized_docs = [doc.split() for doc in docs]

    k1, b = 1.5, 0.75

    doc_lengths = [len(doc) for doc in tokenized_docs]
    avg_doc_length = sum(doc_lengths) / len(doc_lengths)

    N = len(docs)
    idf = {}
    for token in tokenized_query:
        n = sum(1 for doc in tokenized_docs if token in doc)
        idf[token] = math.log((N - n + 0.5) / (n + 0.5) + 1)

    scores = []
    for doc_id, doc in enumerate(tokenized_docs):
      score = 0
      doc_length = doc_lengths[doc_id]
      for token in tokenized_query:
        if token not in doc:
          continue
        tf = doc.count(token)
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * doc_length / avg_doc_length)
        score += idf[token] * (numerator / denominator)
      scores.append(score)

    return np.array(scores)

  def prepare_contexts(self, dataset):
    if "train" in dataset and len(dataset["train"]) > 0:
      contexts = [self.normalize_text(item["context"])
                  for item in dataset["train"]
                  if item["context"].strip()]

      self.contexts = list(dict.fromkeys(contexts))

      all_embeddings = []
      for i in range(0, len(self.contexts), self.batch_size):
        batch = self.contexts[i : i + self.batch_size]
        with torch.no_grad():
          embeddings = self.retriever(batch)
          batch_embeddings = [emb[0][0] for emb in embeddings]
          all_embeddings.extend(batch_embeddings)

      self.context_embeddings = torch.stack([torch.tensor(emb) for emb in all_embeddings])
      self.context_embeddings = F.normalize(self.context_embeddings, p=2, dim=1)


  def find_best_contexts(self, question: str, k: int = 2) -> List[Tuple[str, float]]:
    if not question.strip():
      return []

    normalized_question = self.normalize_text(question)

    with torch.no_grad():
      question_embedding = self.retriever(normalized_question)[0][0]
      question_embedding = F.normalize(torch.tensor(question_embedding).unsqueeze(0), p=2, dim=1)
      semantic_scores = torch.matmul(question_embedding, self.context_embeddings.T).squeeze()

      bm25_scores = self.calculate_bm25_scores(normalized_question, self.contexts)

      semantic_scores = (semantic_scores - semantic_scores.min()) / (semantic_scores.max() - semantic_scores.min())
      bm25_scores = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-6)

      combined_scores = 0.6 * semantic_scores.numpy() + 0.4 * bm25_scores

      top_k_idx = np.argsort(combined_scores)[-k:][::-1]
      results = [(self.contexts[idx], float(combined_scores[idx])) for idx in top_k_idx]

      return results


  def find_best_context(self, question: str, max_tokens: int = 512) -> str:
    contexts = self.find_best_contexts(question, k=2)
    if not contexts:
      return ""

    combined = ""
    tokens = 0
    for context, _ in contexts:
      context_tokens = len(context.split())
      if tokens + context_tokens > max_tokens:
        break
      combined += " " + context
      tokens += context_tokens

    return combined.strip()

In [94]:
class QATrainer(Trainer):
  def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
    outputs = model(**inputs)
    loss = outputs.loss
    return (loss, outputs) if return_outputs else loss

In [95]:
def load_data():
    ds = load_dataset("json", data_files="/content/QAs.json")
    return ds

In [96]:
def setup_system(model_name="dbmdz/bert-base-turkish-cased"):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = AutoModelForQuestionAnswering.from_pretrained(model_name).to(device)
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  retriever = pipeline("feature-extraction", model=model_name)
  return model, tokenizer, retriever, device

In [97]:
def realign_dataset(dataset):
  new_data = {
      "question": [],
      "context": [],
      "answer": [],
  }

  filtered_empty = 0
  filtered_overlap = 0

  for item in tqdm(dataset["train"]):
    try:
      question = item["question"].strip()
      context = item["context"].strip()
      answer = item["answer"].strip()

      if not question or not context or not answer:
        filtered_empty += 1
        continue

      clean_answer = re.sub(r'^(evet|hayır),?\s*', '', answer.lower())
      clean_answer_words = set(clean_answer.split())

      if not clean_answer_words:
        filtered_empty += 1
        continue

      context_words = context.lower().split()
      best_match = ""
      max_overlap = 0

      window_size = min(len(clean_answer_words) + 1, 15)

      if window_size > len(context_words):
        window_size = len(context_words)

      for j in range(len(context_words) - window_size + 1):
        window = ' '.join(context_words[j : j + window_size])
        window_words = set(window.split())
        overlap = len(clean_answer_words & window_words)
        if overlap > max_overlap:
          max_overlap = overlap
          best_match = window

      overlap_ratio = max_overlap / len(clean_answer_words) if clean_answer_words else 0
      if overlap_ratio < 0.5:
        filtered_overlap += 1
      else:
        final_answer = best_match.strip() if best_match and len(best_match.split()) <= 15 else answer
        new_data["question"].append(question)
        new_data["context"].append(context)
        new_data["answer"].append(final_answer)

    except Exception as e:
      print(f"\nHata oluştu: {str(e)}")
      print(f"Probleme neden olan item: {str(item)[:200]}...")
      continue

  return new_data

In [98]:
def preprocess_data(data, tokenizer, max_length=512, stride=128):
  questions = [q.strip() for q in data["question"]]
  contexts = [c.strip() for c in data["context"]]
  answers = [a.strip() for a in data["answer"]]

  all_inputs = {
    "input_ids": [],
    "attention_mask": [],
    "start_positions": [],
    "end_positions": [],
    "overflow_to_sample_mapping": [],
  }

  for i, (question, context, answer) in enumerate(zip(questions, contexts, answers)):
    if not context or not answer:
      continue

    context = re.sub(r'\s+', ' ', context.lower().strip())
    answer = re.sub(r'\s+', ' ', answer.lower().strip())

    start_char = -1
    best_ratio = 0
    for j in range(len(context)):
      if j + len(answer) > len(context):
        break
      current = context[j : j + len(answer)]
      current_ratio = fuzz.ratio(current, answer)
      if current_ratio > best_ratio and current_ratio > 85:
        best_ratio = current_ratio
        start_char = j

    if start_char == -1:
      continue

    end_char = start_char + len(answer)

    tokenized_examples = tokenizer(
        question,
        context,
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = tokenized_examples["offset_mapping"]
    sample_mapping = tokenized_examples["overflow_to_sample_mapping"]

    for split_idx, offsets in enumerate(offset_mapping):
      sample_index = sample_mapping[split_idx]

      context_start = 0
      while context_start < len(offsets) and offsets[context_start][0] is None:
        context_start += 1

      context_end = len(offsets) - 1
      while context_end >= 0 and offsets[context_end][1] is None:
        context_end -= 1

      if start_char < offsets[context_start][0] or end_char > offsets[context_end][1]:
          start_position = 0
          end_position = 0
      else:
          idx = context_start
          while idx <= context_end and offsets[idx][0] <= start_char:
              idx += 1
          start_position= idx - 1

          idx = context_end
          while idx >= context_start and offsets[idx][1] >= end_char:
              idx -= 1
          end_position = idx + 1

          if end_position < start_position:
              start_position = 0
              end_position = 0

      all_inputs["input_ids"].append(tokenized_examples["input_ids"][split_idx])
      all_inputs["attention_mask"].append(tokenized_examples["attention_mask"][split_idx])
      all_inputs["start_positions"].append(start_position)
      all_inputs["end_positions"].append(end_position)
      all_inputs["overflow_to_sample_mapping"].append(sample_index)

  for key in ["input_ids", "attention_mask"]:
    if all_inputs[key]:
      all_inputs[key] = torch.stack([torch.tensor(x) for x in all_inputs[key]])
    else:
      print(f"Warning: {key} is empty, skipping torch.stack")

  if all_inputs["start_positions"]:
    all_inputs["start_positions"] = torch.tensor(all_inputs["start_positions"])
    all_inputs["end_positions"] = torch.tensor(all_inputs["end_positions"])

  return all_inputs

In [133]:
def fine_tune_model(model, tokenizer, dataset, context_retriever, output_dir="/content"):
  def preprocess_with_context(examples):
    contexts = []
    for q in examples["question"]:
      context = context_retriever.find_best_context(q)
      contexts.append(context)

    examples["context"] = contexts

    return preprocess_data(examples, tokenizer)


  tokenized_dataset = dataset.map(
      preprocess_with_context,
      batched=True,
      remove_columns=dataset["train"].column_names,
  )

  tokenized_dataset = tokenized_dataset.filter(
      lambda x: len(x["input_ids"]) > 0,
      desc="Filtering empty examples",
  )

  if len(tokenized_dataset["train"]) == 0:
    raise ValueError("Preprocessing sonrası dataset boş!")

  training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir=output_dir,
    logging_steps=10,
  )

  trainer = QATrainer(
      model=model,
      args=training_args,
      train_dataset=tokenized_dataset["train"],
      eval_dataset=tokenized_dataset["test"],
      tokenizer=tokenizer,
      compute_metrics=lambda x: compute_metrics(x, tokenizer, dataset['test']),
  )

  trainer.train()
  return trainer

In [100]:
def normalize_answer(s):
  def remove_punc(text):
    exclude = set(string.punctuation + '"'"'")
    return ''.join(ch for ch in text if ch not in exclude)


  def white_space_fix(text):
    return ' '.join(text.split())


  def lower(text):
    return text.lower().replace('I', 'ı').replace('İ', 'i')


  return white_space_fix(remove_punc(lower(s.strip())))

In [101]:
def compute_em(prediction, truth):
  return int(normalize_answer(prediction) == normalize_answer(truth))

In [102]:
def compute_f1(prediction, truth):
  prediction_tokens = normalize_answer(prediction).split()
  truth_tokens = normalize_answer(truth).split()

  if len(prediction_tokens) == 0 or len(truth_tokens) == 0:
    return int(prediction_tokens == truth_tokens)

  common_tokens = Counter(prediction_tokens) & Counter(truth_tokens)

  if len(common_tokens) == 0:
    return 0

  prec = len(common_tokens) / len(prediction_tokens)
  rec = len(common_tokens) / len(truth_tokens)
  f1 = 2 * (prec * rec) / (prec + rec)

  return f1

In [135]:
def compute_metrics(eval_pred, tokenizer, dataset):
  predictions, labels = eval_pred
  start_logits, end_logits = predictions
  examples = dataset

  inputs = tokenizer(
      [ex["question"] for ex in examples],
      [ex["context"] for ex in examples],
      truncation=True,
      max_length=512,
      return_tensors="pt",
      padding=True,
  )

  ems, f1s = [], []
  for i, (start, end) in enumerate(zip(start_logits, end_logits)):
    start_idx = np.argmax(start)
    end_idx = np.argmax(end)

    pred_text = tokenizer.decode(
        inputs["input_ids"][i][start_idx : end_idx + 1],
        skip_special_tokens=True,
    )
    true_text = examples[i]["answer"]

    ems.append(compute_em(pred_text, true_text))
    f1s.append(compute_f1(pred_text, true_text))

  return {
      "em": np.mean(ems),
      "f1": np.mean(f1s),
  }

In [139]:
def ask_question(model, tokenizer, context_retriever):
  print("Çıkmak için 'çıkış' yazın.")

  while True:
    question = input("Soru: ")
    if question.lower() == "çıkış":
      break

    context = context_retriever.find_best_context(question)
    if not context:
      print("Uygun bir bağlam bulunamadı, daha spesifik bir soru sormayı deneyin.")
      continue

    inputs = tokenizer(
        question,
        context,
        add_special_tokens=True,
        return_tensors="pt",
        max_length=512,
        padding="max_length",
        truncation=True,
    ).to(model.device)

    outputs = model(**inputs)

    start_probs = torch.nn.functional.softmax(outputs.start_logits, dim=1)
    end_probs = torch.nn.functional.softmax(outputs.end_logits, dim=1)

    start_idx = torch.argmax(start_probs, dim=1).item()
    end_idx = torch.argmax(end_probs, dim=1).item()

    if end_idx < start_idx:
        end_idx = start_idx

    answer_tokens = inputs.input_ids[0, start_idx:end_idx + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)

    print(f"Context: {context}")

    if not answer.strip():
        print("Cevap bulunamadı.")
        continue

    print(f"Cevap: {answer}")

In [108]:
def train_and_run_model():
  model, tokenizer, retriever, device = setup_system()

  dataset = load_data()

  context_retriever = SimpleContextRetriever(retriever)
  context_retriever.prepare_contexts(dataset)

  aligned_dataset = realign_dataset(dataset)

  aligned_dataset = Dataset.from_dict({
      "question": aligned_dataset["question"],
      "context": aligned_dataset["context"],
      "answer": aligned_dataset["answer"],
  })

  split_dataset = aligned_dataset.train_test_split(test_size=0.2)

  trainer = fine_tune_model(model, tokenizer, split_dataset, context_retriever)

  trained_model = trainer.model

  ask_question(trained_model, tokenizer, context_retriever)

In [ ]:
if __name__ == "__main__":
  train_and_run_model()

In [ ]:
trainer.save_model("/content/drive/MyDrive/fine_tuned_model")
tokenizer.save_pretrained("/content/drive/MyDrive/fine_tuned_model")